In [1]:
import json
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
import matplotlib.pyplot as plt
from collections import Counter
import os

PROCESSED_DIR = "../data/processed/"
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", DEVICE)

# Load BIO labeled data
with open(os.path.join(PROCESSED_DIR, "bio_labeled_sample.json")) as f:
    records = json.load(f)

print(f"Loaded {len(records)} records")
print("\nSample record keys:", records[0].keys())
print("Sample words:", records[0]['words'][:8])
print("Sample labels:", records[0]['labels'][:8])

Using device: cpu
Loaded 30000 records

Sample record keys: dict_keys(['drug', 'condition', 'rating', 'review', 'words', 'labels'])
Sample words: ['"Worked', 'fine', 'for', '2', 'days', '-', 'burning', 'and']
Sample labels: ['O', 'O', 'O', 'O', 'O', 'O', 'B-ADR', 'O']


In [2]:
def bio_to_binary(labels):
    """If any ADR token exists in review → label 1, else 0"""
    return 1 if any(l!='O' for l in labels) else 0

texts= [" ".join(r["words"]) for r in records]
labels = [bio_to_binary(r["labels"]) for r in records]

print(f"Positive ADR: {sum(labels):,} ({sum(labels)/len(labels):.1%})")
print(f"Negative ADR: {len(labels) - sum(labels):,} ({(len(labels) - sum(labels))/len(labels):.1%})")


Positive ADR: 29,296 (97.7%)
Negative ADR: 704 (2.3%)


In [3]:
# Tokenize all texts
tokenized_text = [text.lower().split() for text in texts]

# Count word frequencies
word_freq = Counter(word for tokens in tokenized_text for word in tokens)
print(f"Total unique words: {len(word_freq):,}")

# Keep only words appearing at least 2 times — reduces noise
MIN_FREQ = 2
vocab = [word for word, count in word_freq.items() if count>= MIN_FREQ]
print(f"Vocab size after min_freq filter: {len(vocab):,}")

# Build word → index mapping
# PAD = padding token (makes all sequences same length)
# UNK = unknown token (for words not in vocab)
word2idx = {"<PAD>": 0, "<UNK>": 1}
for word in sorted(vocab):
    word2idx[word]= len(word2idx)

idx2word = {v: k for k, v in word2idx.items()}
VOCAB_SIZE = len(word2idx)
print(f"Final vocab size (with PAD+UNK): {VOCAB_SIZE:,}")    

Total unique words: 76,263
Vocab size after min_freq filter: 34,957
Final vocab size (with PAD+UNK): 34,959


In [4]:
print(word2idx)

{'<PAD>': 0, '<UNK>': 1, '!': 2, '!!': 3, '!!!': 4, '!!!!': 5, '!!!!!': 6, '!!!!!"': 7, '!!!!"': 8, '!!!"': 9, '!!"': 10, '!"': 11, '!.': 12, '"': 13, '""': 14, '"(15': 15, '"*update': 16, '".': 17, '"1': 18, '"10': 19, '"10"': 20, '"10-15': 21, '"12': 22, '"12.5': 23, '"13': 24, '"15': 25, '"16': 26, '"17': 27, '"18mg': 28, '"19': 29, '"1st': 30, '"2': 31, '"20': 32, '"22': 33, '"23': 34, '"23,': 35, '"24': 36, '"25': 37, '"26': 38, '"27': 39, '"28': 40, '"2mg': 41, '"2nd': 42, '"3': 43, '"30/f.': 44, '"34': 45, '"35': 46, '"36': 47, '"37': 48, '"38': 49, '"39': 50, '"3rd': 51, '"4': 52, '"40': 53, '"42': 54, '"45': 55, '"46': 56, '"49': 57, '"5': 58, '"5\'': 59, '"50': 60, '"51': 61, '"52': 62, '"53': 63, '"54': 64, '"55': 65, '"57': 66, '"59': 67, '"5mg': 68, '"5th': 69, '"6': 70, '"65': 71, '"7': 72, '"8': 73, '"9': 74, '"9/10': 75, '"a': 76, '"abilify': 77, '"about': 78, '"abreva': 79, '"absolute': 80, '"absolutely': 81, '"accutane': 82, '"aches': 83, '"acne': 84, '"activating"': 

In [5]:
MAX_LEN = 200   # truncate/pad all reviews to this length

class DrugReviewDataset(Dataset):
    def __init__(self, texts, labels, word2idx, max_len):
        self.labels = labels
        self.max_len = max_len
        # Convert each text to a padded tensor of integer IDs
        self.encoded = [self.encode(text, word2idx) for text in texts]
    
    def encode(self, text, word2idx):
        tokens = text.lower().split()[:self.max_len]   # truncate
        ids = [word2idx.get(t, 1) for t in tokens]    # 1 = UNK
        # Pad to max_len
        ids += [0] * (self.max_len - len(ids))        # 0 = PAD
        return torch.tensor(ids, dtype=torch.long)
    
    def __len__(self):
        return len(self.labels)
    
    def __getitem__(self, idx):
        return self.encoded[idx], torch.tensor(self.labels[idx], dtype=torch.float)

In [7]:
# Train / val / test split — 70 / 15 / 15
x_train, x_temp, y_train, y_temp = train_test_split(texts, labels, test_size =0.3, stratify= labels, random_state = 42)
x_val, x_test, y_val, y_test = train_test_split(texts, labels, test_size = 0.5, stratify = labels, random_state =42)

print(f"Train: ", len(x_train),"| Val: ", len(x_val), "| Test: ", len(x_test))

# Create datasets
train_ds= DrugReviewDataset(x_train, y_train, word2idx, MAX_LEN)
val_ds= DrugReviewDataset(x_val, y_val, word2idx, MAX_LEN)
test_ds= DrugReviewDataset(x_test, y_test, word2idx, MAX_LEN)

# Create loaders
train_loader = DataLoader(train_ds, batch_size = 32, shuffle = True)
val_loader = DataLoader(val_ds, batch_size = 32, shuffle = True)
test_loader = DataLoader(test_ds, batch_size = 32, shuffle = True)

print(f"\nBatches per epoch: {len(train_loader)}")

Train:  21000 | Val:  15000 | Test:  15000

Batches per epoch: 657


In [14]:
class LSTMClassifier(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, n_layers, dropout):
        super().__init__()
        
        # Layer 1: embedding — converts integer IDs to dense vectors
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx = 0)

        # Layer 2: LSTM — processes sequence, returns hidden states
        self.lstm = nn.LSTM(embed_dim, 
                         hidden_dim, 
                         num_layers= n_layers, 
                         batch_first= True, 
                         dropout= dropout if n_layers > 1 else 0,
                        bidirectional=True      # reads left→right AND right→left
                        )
        # Layer 3: classifier head
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden_dim * 2, 1)

    def forward(self, x):
       # x shape= (batch_size, seq_len)
        embedded = self.dropout(self.embedding(x))
        # embedded shape: (batch_size, seq_len, embed_dim)

        lstm_out, (hidden, _) = self.lstm(embedded)
        # hidden shape: (n_layers * 2, batch_size, hidden_dim)
        hidden =torch.cat((hidden[-2], hidden[-1]), dim=1)
        # hidden shape: (batch_size, hidden_dim * 2)

        out=  self.fc(self.dropout(hidden))
        # out shape: (batch_size, 1)
        return out.squeeze(1)

In [18]:
# Initialize model
model = LSTMClassifier(
    vocab_size= VOCAB_SIZE,
    embed_dim= 200,
    hidden_dim= 128,
    n_layers= 2,
    dropout=0.3
).to(DEVICE)

print(model)
print(f"\nTrainable parameters:  {sum(p.numel() for p in model.parameters()):,}")

optimizer = torch.optim.Adam(model.parameters(), lr= 1e-3)
criterion = nn.BCEWithLogitsLoss()   # handles sigmoid internally — more stable 

def train_epoch(model, loader, optimizer, criterion):
    model.train()
    total_loss, correct, total = 0, 0, 0
    for inputs, targets in loader:
        inputs, targets = inputs.to(DEVICE), targets.to(DEVICE)
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, targets)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        total_loss += loss.item()
        preds = (torch.sigmoid(outputs) > 0.5).float()
        correct += (preds == targets).sum().item()
        total += targets.size(0)
    return total_loss / len(loader), correct / total


def evaluate(model, loader, criterion):
    model.eval()
    total_loss, correct, total = 0, 0, 0
    with torch.no_grad():
        for inputs, targets in loader:
            inputs, targets = inputs.to(DEVICE), targets.to(DEVICE)
            outputs = model(inputs)
            loss = criterion(outputs, targets)
            total_loss += loss.item()
            preds = (torch.sigmoid(outputs) > 0.5).float()
            correct += (preds == targets).sum().item()
            total += targets.size(0)
    return total_loss / len(loader), correct / total

LSTMClassifier(
  (embedding): Embedding(34959, 200, padding_idx=0)
  (lstm): LSTM(200, 128, num_layers=2, batch_first=True, dropout=0.3, bidirectional=True)
  (dropout): Dropout(p=0.3, inplace=False)
  (fc): Linear(in_features=256, out_features=1, bias=True)
)

Trainable parameters:  7,725,241


In [19]:
EPOCHS =5
train_losses, val_losses =[],[]

for epoch in range(EPOCHS):
    train_loss, train_acc = train_epoch(model, train_loader, optimizer, criterion)
    val_loss, val_acc = evaluate(model, val_loader, criterion)

    train_losses.append(train_loss)
    val_losses.append(val_loss)

    print(f"Epoch {epoch+1}/{EPOCHS} | "
          f"Train loss: {train_loss:.4f} acc: {train_acc:.3f} | "
          f"Val loss: {val_loss:.4f} acc: {val_acc:.3f}")

Epoch 1/5 | Train loss: 0.1184 acc: 0.975 | Val loss: 0.1069 acc: 0.977
Epoch 2/5 | Train loss: 0.1059 acc: 0.976 | Val loss: 0.0969 acc: 0.977
Epoch 3/5 | Train loss: 0.0890 acc: 0.978 | Val loss: 0.0607 acc: 0.982
Epoch 4/5 | Train loss: 0.0678 acc: 0.982 | Val loss: 0.0448 acc: 0.988
Epoch 5/5 | Train loss: 0.0536 acc: 0.985 | Val loss: 0.0349 acc: 0.989


In [20]:
from sklearn.metrics import classification_report

model.eval()
all_preds, all_targets = [], []

with torch.no_grad():
    for inputs, targets in test_loader:
        inputs= inputs.to(DEVICE)
        outputs = model(inputs)
        preds = (torch.sigmoid(outputs)>0.5).float().cpu()
        all_preds.extend(preds.numpy())
        all_targets.extend(targets.numpy())

print(classification_report(
    all_targets, all_preds,
    target_names=['No ADR', 'ADR'],
    digits=3
))

              precision    recall  f1-score   support

      No ADR      0.852     0.310     0.454       352
         ADR      0.984     0.999     0.991     14648

    accuracy                          0.983     15000
   macro avg      0.918     0.654     0.723     15000
weighted avg      0.981     0.983     0.979     15000



In [21]:
# Save model and vocabulary
torch.save(model.state_dict(), "../models/lstm_classifier.pt")

import json
with open("../models/word2idx.json", "w") as f:
    json.dump(word2idx, f)

print("LSTM model and vocabulary saved")

LSTM model and vocabulary saved
